In [98]:
%pip install langgraph langchain langchain-ollama requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [99]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama

import requests
import json
import requests
from typing import TypedDict
import gradio as gr

In [100]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

In [101]:

API_KEY = "24ffeabd81117e914b56fd18c87d68cc"   

def get_weather(city):
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

    response = requests.get(url)
    data = response.json()

    return {
        "city": city,
        "temperature": data["main"]["temp"],
        "humidity": data["main"]["humidity"],
        "condition": data["weather"][0]["main"]
    }

In [102]:


def heater_on():
    heater_status = "ON"
    print("🔥 Heater turned ON")
    return heater_status


def heater_off():
    heater_status = "OFF"
    print("❄️ Heater turned OFF")
    return heater_status

In [103]:

class WeatherState(TypedDict):
    city: str
    weather: dict
    heater_status: str
    decision: str

In [104]:
def weather_node(state: WeatherState):
    state["weather"] = get_weather(state["city"])
    return state

In [105]:
def decision_node(state: WeatherState):
    weather = state["weather"]

    prompt = f"""
    You are a smart home AI.

    Current Weather:
    Temperature: {weather['temperature']}°C
    Humidity: {weather['humidity']}%
    Condition: {weather['condition']}

    Decide whether the heater should be ON or OFF.

    Rules:
    - If temperature is below 18°C, answer ON.
    - Otherwise answer OFF.

    IMPORTANT:
    Respond with ONLY one word:
    ON
    or
    OFF
    """

    response = llm.invoke(prompt)

    state["decision"] = response.content.strip().upper()

    return state

In [106]:
def heater_node(state: WeatherState):
    if state["decision"] == "ON":
        state["heater_status"] = heater_on()
    else:
        state["heater_status"] = heater_off()

    return state

In [107]:
builder = StateGraph(WeatherState)

builder.add_node("weather", weather_node)
builder.add_node("decision", decision_node)
builder.add_node("heater", heater_node)

builder.add_edge(START, "weather")
builder.add_edge("weather", "decision")
builder.add_edge("decision", "heater")
builder.add_edge("heater", END)

graph = builder.compile()

In [108]:
result = graph.invoke({
    "city": "Mumbai"
})

print(result)

❄️ Heater turned OFF
{'city': 'Mumbai', 'weather': {'city': 'Mumbai', 'temperature': 28.31, 'humidity': 80, 'condition': 'Clouds'}, 'heater_status': 'OFF', 'decision': 'OFF'}


In [109]:
def weather_agent(city):

    result = graph.invoke({
        "city": city
    })

    weather = result["weather"]

    recommendation = (
        "🔥 Heater should be ON"
        if result["heater_status"] == "ON"
        else "❄️ Heater should be OFF"
    )

    return (
        weather["city"],
        weather["temperature"],
        weather["humidity"],
        weather["condition"],
        result["heater_status"],
        recommendation,
    )

In [110]:
demo = gr.Interface(
    fn=weather_agent,

    inputs=gr.Textbox(
        label="Enter City",
        placeholder="Mumbai"
    ),

    outputs=[
        gr.Textbox(label="City"),
        gr.Number(label="Temperature (°C)"),
        gr.Number(label="Humidity (%)"),
        gr.Textbox(label="Condition"),
        gr.Textbox(label="Heater"),
        gr.Textbox(label="AI Recommendation"),
    ],

    title="🌤️ Weather Agentic AI",
    description="Uses LangGraph + Llama 3.2 + OpenWeatherMap API",
)

In [ ]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "e:\Python\Lib\site-packages\gradio\queueing.py", line 763, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Python\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Python\Lib\site-packages\gradio\blocks.py", line 2106, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Python\Lib\site-packages\gradio\blocks.py", line 1588, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Python\Lib\site-packages\anyio\to_thread.py", line 61, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\P

❄️ Heater turned OFF


Traceback (most recent call last):
  File "e:\Python\Lib\site-packages\gradio\queueing.py", line 763, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Python\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Python\Lib\site-packages\gradio\blocks.py", line 2106, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Python\Lib\site-packages\gradio\blocks.py", line 1588, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Python\Lib\site-packages\anyio\to_thread.py", line 61, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\P

❄️ Heater turned OFF
❄️ Heater turned OFF
❄️ Heater turned OFF
❄️ Heater turned OFF
